# LED Matrix CNN: Predict 8×8 Board State from Images

This notebook trains a convolutional neural network (CNN) that takes a picture of your 8×8 LED matrix and predicts the corresponding 64‑element board array (0 = off, 1 = red, 2 = blue).

**Expected data layout**

- A folder of images named `1`, `2`, ..., `N` (for example `1.png`, `2.jpg`, ...).
- A CSV file with `N` rows and 64 columns. Row *i* is the board state for image *i*.

In Google Colab you can:
1. Upload your `photos.zip` (or image folder) and your `boards.csv` to `/content`.
2. Run the cells below in order to train the model.


In [ ]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split

print('TensorFlow version:', tf.__version__)


In [ ]:
# ==== Paths and basic settings ====

# Where your image files will live in Colab.
# If you upload a `photos.zip` file, you can unzip it into this folder.
IMG_DIR = '/content/photos'

# Path to the CSV that contains the 64-element board arrays.
CSV_PATH = '/content/boards.csv'

# Image size that we will resize all images to.
IMG_HEIGHT = 128
IMG_WIDTH = 128

NUM_CLASSES = 3  # 0 = off, 1 = red, 2 = blue

# If you uploaded a photos.zip file, uncomment the line below once and run it
# to unzip the images into IMG_DIR.
# !unzip -q '/content/photos.zip' -d '/content/photos'


In [ ]:
# ==== Load the CSV of board states ====

df = pd.read_csv(CSV_PATH, header=None)

print('CSV shape (rows, columns):', df.shape)

# Keep only the first 64 columns in case there is extra metadata.
if df.shape[1] > 64:
    df = df.iloc[:, :64]

if df.shape[1] != 64:
    raise ValueError(f'Expected 64 columns for the board state, but got {df.shape[1]}.')

boards_all = df.values.astype('int64')
num_samples = boards_all.shape[0]

print('Number of samples (rows):', num_samples)
print('Example board row (first row):')
print(boards_all[0])


In [ ]:
# ==== Load and preprocess images ====

from tensorflow.keras.utils import load_img, img_to_array

def find_image_path(idx, img_dir=IMG_DIR):
    """Given an index (1-based), try to find the corresponding image file
    with common extensions. Returns None if nothing is found.
    """
    idx_str = str(idx)

    # Try plain names like '1.png', '1.jpg', ...
    exts = ['.png', '.jpg', '.jpeg', '.bmp']
    for ext in exts:
        path = os.path.join(img_dir, idx_str + ext)
        if os.path.exists(path):
            return path

    # Try zero-padded names like '001.png', '0001.jpg', ...
    for width in [3, 4, 5]:
        for ext in exts:
            path = os.path.join(img_dir, f"{idx:0{width}d}" + ext)
            if os.path.exists(path):
                return path

    return None

images = []
labels = []
missing_indices = []

for i in range(1, num_samples + 1):
    img_path = find_image_path(i)
    if img_path is None:
        missing_indices.append(i)
        continue

    img = load_img(img_path, target_size=(IMG_HEIGHT, IMG_WIDTH))
    img_array = img_to_array(img) / 255.0  # normalize to [0, 1]
    images.append(img_array)
    labels.append(boards_all[i - 1])

images = np.array(images, dtype='float32')
labels = np.array(labels, dtype='int64')

print('Loaded image tensor shape:', images.shape)
print('Loaded labels shape:', labels.shape)
print('Number of missing image indices:', len(missing_indices))
if missing_indices:
    print('First few missing indices:', missing_indices[:10])


In [ ]:
# ==== Train / validation split ====

X_train, X_val, y_train, y_val = train_test_split(
    images,
    labels,
    test_size=0.2,
    random_state=42,
)

print('Train images:', X_train.shape, 'Train labels:', y_train.shape)
print('Val images:', X_val.shape, 'Val labels:', y_val.shape)


In [ ]:
# ==== Build the CNN model ====

input_shape = (IMG_HEIGHT, IMG_WIDTH, 3)

inputs = keras.Input(shape=input_shape)

x = layers.Conv2D(32, 3, activation='relu', padding='same')(inputs)
x = layers.MaxPooling2D()(x)

x = layers.Conv2D(64, 3, activation='relu', padding='same')(x)
x = layers.MaxPooling2D()(x)

x = layers.Conv2D(128, 3, activation='relu', padding='same')(x)
x = layers.MaxPooling2D()(x)

x = layers.Flatten()(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.3)(x)

# Final Dense layer has 64 * NUM_CLASSES units.
x = layers.Dense(64 * NUM_CLASSES)(x)
x = layers.Reshape((64, NUM_CLASSES))(x)

# Softmax over the 3 classes at each of the 64 positions.
outputs = layers.Activation('softmax')(x)

model = keras.Model(inputs=inputs, outputs=outputs, name='led_matrix_cnn')

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',  # y is (batch, 64) with integer labels 0, 1, 2
    metrics=['accuracy'],
)

model.summary()


In [ ]:
# ==== Train the model ====

batch_size = 16
epochs = 30

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
    )
]

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    batch_size=batch_size,
    epochs=epochs,
    callbacks=callbacks,
)


In [ ]:
# ==== Plot training curves ====

import matplotlib.pyplot as plt

acc = history.history.get('accuracy', [])
val_acc = history.history.get('val_accuracy', [])
loss = history.history.get('loss', [])
val_loss = history.history.get('val_loss', [])

epochs_range = range(1, len(acc) + 1)

plt.figure()
plt.plot(epochs_range, acc, label='Train accuracy')
plt.plot(epochs_range, val_acc, label='Val accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.show()

plt.figure()
plt.plot(epochs_range, loss, label='Train loss')
plt.plot(epochs_range, val_loss, label='Val loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# ==== Evaluate and inspect predictions ====

val_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)
print(f'Validation loss: {val_loss:.4f}')
print(f'Validation accuracy (per LED cell): {val_acc:.4f}')

# Pick one example from the validation set
idx = 0
x_sample = X_val[idx:idx + 1]
y_true = y_val[idx]

y_pred_probs = model.predict(x_sample)
y_pred = np.argmax(y_pred_probs, axis=-1)[0]  # shape (64,)

print('True board (first validation example):')
print(y_true.reshape(8, 8))

print('\nPredicted board:')
print(y_pred.reshape(8, 8))


In [ ]:
# ==== Save the model and define a convenience prediction function ====

MODEL_PATH = '/content/led_matrix_cnn_model'

model.save(MODEL_PATH)
print('Saved model to:', MODEL_PATH)

def predict_board_from_path(image_path):
    """Helper function that loads an image of the LED matrix and
    returns the predicted 64-element board array.
    """
    img = load_img(image_path, target_size=(IMG_HEIGHT, IMG_WIDTH))
    img_array = img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    probs = model.predict(img_array)
    board_pred = np.argmax(probs, axis=-1)[0]  # shape (64,)
    return board_pred

# Example (change the path to any image you want to test):
# example_path = os.path.join(IMG_DIR, '1.png')
# print(predict_board_from_path(example_path))
